# data_access

> Functionality to access Euclid calibrated frames, mosaic tiles and raw data for specific targets and observations.

In [ ]:
# | default_exp euclid.data_access

In [ ]:
# | export

import re
from getpass import getpass
import os
from pathlib import Path

import numpy as np
import pandas as pd
from astropy import table
from astropy.table import Table
from astroquery.utils.tap.core import TapPlus

from nicl.euclid.utilities import default_data_path, euclid_credentials
from nicl.utilities import maybe_to_value

In [ ]:
# | hide
from nbdev import show_doc

In [ ]:
# | export


class DataAccess:
    """Provides access to Euclid data."""

    def __init__(
        self,
        esa_username=None,  # ESA account username (prompts if not supplied)
        esa_password=None,  # ESA account password (prompts if not supplied)
        esac_server_url="https://easidr.esac.esa.int",  # ESA server (default is Q1)
        release_name="Q1_R1",  # the Euclid release name
        dry_run=False,  # if True, do not actually download files
        overwrite=False,  # should existing files be overwritten?
    ):
        """Create an object for accessing data and log in to the ESA server."""
        credentials = euclid_credentials()
        if credentials is not None:
            if esa_username is None:
                esa_username = credentials["user"]
            if esa_password is None:
                esa_password = credentials["password"]
        if esa_username is None:
            esa_username = getpass(prompt="ESA User:")
        if esa_password is None:
            esa_password = getpass(prompt="ESA Password:")
        self.esa_username = esa_username
        self.esa_password = esa_password
        self.release_name = release_name
        self.dry_run = dry_run
        self.overwrite = overwrite
        self.tap = TapPlus(url=f"{esac_server_url}/tap-server", tap_context="tap")
        self.data_tap = TapPlus(url=f"{esac_server_url}/sas-dd", data_context="data")
        self.mer_filename_lookup = self.get_mer_filename_lookup()

    def tap_login(self):
        self.tap.login(user=self.esa_username, password=self.esa_password)

    def data_login(self):
        self.data_tap.login(user=self.esa_username, password=self.esa_password)

    def tap_query(self, query):
        self.tap_login()
        job = self.tap.launch_job(query)
        return job.get_results()

    def get_mer_filename_lookup(self):
        """Build a DataFrame of MER files.

        This assumes there is a file named "filelist" in the MER folder, containing
        a listing of the MER files in archive, created by running `find .` in the
        MER folder in DataLabs. This is a workaround, due to the normal ways of
        accessing non-STK MER files not currently working.
        """

        def get_mer_info(row):
            fn = row["filename"]
            matched = re.search("EUC_MER_(.+)_TILE", fn)[1]
            if matched.endswith("RMS") or matched.endswith("FLAG"):
                match = re.search("MOSAIC-(.+)-(RMS|FLAG)", matched)
                filt, file_type = match.groups()
            else:
                match = re.search("(.+)-(.+-?.)", matched)
                file_type, filt = match.groups()
            if file_type == "BGSUB-MOSAIC":
                t = "STK"
            elif file_type == "BGMOD":
                t = "BKG"
            else:
                t = file_type
            filt = filt.replace("-", "_")
            return t, filt

        path = default_data_path("Q1_R1", "MER")
        fn = path / "filelist"
        if fn.exists():
            table = pd.read_table(
                fn,
                header=None,
                names=["dot", "tile_index", "instrument", "filename"],
                dtype=str,
                sep="/",
            )
            table = table.drop(columns="dot").dropna()
            table = table[table["filename"].str.startswith("EUC")]
            cols = table.apply(get_mer_info, axis="columns", result_type="expand")
            cols.columns = ["mer_file_type", "filter"]
            table = pd.concat((table, cols), axis="columns")
            table = table.groupby(
                ["tile_index", "instrument", "mer_file_type", "filter"]
            ).first()
            table = table["filename"]
            return table

    def build_instrument_condition(self, instrument, filter, raw=False):
        release_condition = "1=1" if raw else f"(release_name='{self.release_name}')"
        instrument_condition = (
            f"AND instrument_name = '{instrument}'" if instrument is not None else ""
        )
        filter_condition = f"AND filter_name = '{filter}'" if filter is not None else ""
        return " ".join((release_condition, instrument_condition, filter_condition))

    def build_fov_condition(self, ra, dec, radius, fully_contained):
        ra, dec, radius = (maybe_to_value(x, "deg") for x in (ra, dec, radius))
        release_condition = f"(release_name='{self.release_name}')"
        criterion = "CONTAINS" if fully_contained else "INTERSECTS"
        fov_condition = f"AND (fov IS NOT NULL AND {criterion}(CIRCLE('ICRS',{ra},{dec},{radius}),fov)=1)"
        return " ".join((release_condition, fov_condition))

    def find_all_observations(
        self,
    ):  # returns a list of observation_ids
        """Obtain a list of all survey obs_ids for observations in the current release."""
        query = f"""SELECT observation_id
                    FROM sedm.calibrated_frame
                    WHERE (product_type like '%Calibrated%')
                    AND release_name='{self.release_name}'
                    ORDER BY observation_id ASC"""
        results = self.tap_query(query)
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def find_all_tiles(
        self,
    ):  # returns a list of observation_ids
        """Obtain a list of all MER tile_indexes for tiles in the current release."""
        query = f"""SELECT tile_index
                    FROM sedm.mosaic_product
                    WHERE release_name='{self.release_name}'
                    ORDER BY tile_index ASC"""
        results = self.tap_query(query)
        tile_indexes = np.unique(list(results["tile_index"])).astype(int)
        return tile_indexes

    def find_observations_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of observation_ids
        """Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = f"""SELECT observation_id
                    FROM sedm.calibrated_frame
                    WHERE (product_type like '%Calibrated%')
                    AND {condition}
                    ORDER BY observation_id ASC"""
        results = self.tap_query(query)
        obs_ids = np.unique(list(results["observation_id"])).astype(int)
        return obs_ids

    def find_tiles_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
    ):  # returns a list of tile_indexes
        """Obtain a list of survey MER tile_indexes for tiles that entirely contain or intersect the specified target region."""
        condition = self.build_fov_condition(ra, dec, radius, fully_contained)
        query = f"""SELECT tile_index
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    ORDER BY tile_index ASC"""
        results = self.tap_query(query)
        tile_indexes = np.unique(list(results["tile_index"])).astype(int)
        return tile_indexes

    def get_calibrated_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain calibrated file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT observation_stack.observation_id, observation_stack.file_name,
                    observation_stack.instrument_name, observation_stack.filter_name, observation_stack.duration
                    FROM sedm.calibrated_frame
                    AS observation_stack
                    WHERE (product_type like '%Calibrated%')
                    AND {condition}
                    AND (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_sir_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
    ):  # returns a table of file information
        """Obtain SIR file information for obs_id."""
        query = f"""SELECT observation_id, file_name
                    FROM sedm.sir_science_frame
                    WHERE (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_raw_files_for_observation(
        self,
        obs_id,  # observation_id for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, "" for VIS, OPEN for grism, CLOSED for slew dark, Y, J or H
    ):  # returns a table of file information
        """Obtain raw file information for obs_id, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter, raw=True)
        query = f"""SELECT raw_frame.file_name, raw_frame.rawframe_oid, raw_frame.observation_id,
                    raw_frame.instrument_name, raw_frame.data_set_release, raw_frame.filter_name,
                    raw_frame.observation_mode, raw_frame.grism_wheel_pos, raw_frame.cal_block_id,
                    raw_frame.cal_block_variant, raw_frame.ra, raw_frame.dec, raw_frame.obs_time_utc,
                    raw_frame.exposure_time
                    FROM sedm.raw_frame
                    WHERE {condition}
                    AND (observation_id = '{obs_id}')"""
        file_info = self.tap_query(query)
        return file_info

    def get_files_for_tile(
        self,
        tile_index,  # tile_index for which to find files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
    ):  # returns a table of file information
        """Obtain mosaic file information for tile_index, optionally restricted by instrument or filter."""
        condition = self.build_instrument_condition(instrument, filter)
        query = f"""SELECT mosaic_product.tile_index, mosaic_product.file_name, mosaic_product.mosaic_product_oid,
                    mosaic_product.instrument_name, mosaic_product.filter_name, mosaic_product.category,
                    mosaic_product.second_type, mosaic_product.ra, mosaic_product.dec, mosaic_product.technique 
                    FROM sedm.mosaic_product
                    WHERE {condition}
                    AND (tile_index = '{tile_index}')"""
        file_info = self.tap_query(query)
        return file_info

    @staticmethod
    def get_instrument_folder(instrument):
        if instrument == "NISP":
            instrument_folder = "NIR"
        elif instrument == "VIS":
            instrument_folder = "VIS"
        else:
            raise ValueError("The instrument must be NISP or VIS.")
        return instrument_folder

    def download_files(
        self,
        filenames,  #  list of filenames or Table containing "file_name" column
        outpath,  # the folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):
        """Download multiple Euclid filenames to outpath."""
        outpath.mkdir(parents=True, exist_ok=True)
        if isinstance(filenames, Table):
            filenames = filenames["file_name"]
        if verbose:
            print(f"Downloading {len(filenames)} files to {outpath}")
        for fn in filenames:
            if verbose:
                print(f"Downloading {fn}")
            self.download_file(fn, outpath)

    def download_file(
        self,
        filename,  # the filename to download
        outpath,  # the folder in which to save the downloaded files
    ):
        """Download Euclid filename to outpath."""
        params_dict = dict(
            RETRIEVAL_TYPE="FILE", RELEASE="sedm", RETRIEVAL_ACCESS="DIRECT"
        )
        params_dict.update(FILE_NAME=filename)
        outpath = Path(outpath).expanduser()
        outfn = outpath / filename
        self.data_login()
        if not self.dry_run:
            if not outfn.exists() or self.overwrite:
                try:
                    self.data_tap.load_data(params_dict=params_dict, output_file=outfn)
                except (Exception, KeyboardInterrupt):
                    if outfn.exists():
                        os.remove(outfn)
                    raise
            else:
                print(f"File already exists, skipping: {outfn}")

    def download_mosaic_files(
        self,
        file_info,  #  Table containing file information
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        verbose=True,  # print information to the screen
    ):
        """Download multiple Euclid mosaics to outpath."""
        outpath.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {len(file_info)} files to {outpath}")
        filenames = []
        for info in file_info:
            t = info["tile_index"]
            i = info["instrument_name"]
            f = info["filter_name"]
            fn = self.download_mosaic_file(
                t, i, f, outpath, mer_file_type, verbose=verbose
            )
            filenames.append(fn)
        return filenames

    def download_mosaic_file(
        self,
        tile_index,
        instrument,  # NISP or VIS
        filter,  # VIS, NIR_Y, NIR_J or NIR_H
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        verbose=True,  # print information to the screen
    ):
        """Download Euclid mosaic to outpath, using a workaround method."""
        if self.mer_filename_lookup is None:
            raise FileNotFoundError("MER filename lookup file was not found.")
        fn = self.mer_filename_lookup.loc[
            f"{tile_index}", instrument, mer_file_type, filter
        ]
        if verbose:
            print(f"Downloading {fn}")
        self.download_file(fn, outpath)
        return fn

    def download_mosaic_file_alt(
        self,
        tile_index,
        instrument,  # NISP or VIS
        filter,  # VIS, NIR_Y, NIR_J or NIR_H
        outpath,  # the folder in which to save the downloaded files
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        verbose=True,  # print information to the screen
    ):
        """Download Euclid mosaic to outpath, using an alternative method.

        This is currently not supported by SAS for file types other than STK.
        """
        params_dict = dict(RETRIEVAL_TYPE="MOSAIC", RELEASE="sedm")
        params_dict.update(
            TILE_INDEX=tile_index,
            INSTRUMENT=instrument,
            FILTER=filter,
            TYPE=mer_file_type,
        )
        outpath = Path(outpath).expanduser()
        if mer_file_type == "STK":
            filename = f"BGSUB-MOSAIC-{filter}"
        elif mer_file_type == "BKG":
            filename = f"BGMOD-{filter}"
        elif mer_file_type == "RMS":
            filename = f"MOSAIC-{filter}-RMS"
        elif mer_file_type == "FLAG":
            filename = f"MOSAIC-{filter}-FLAG"
        else:
            raise ValueError(f"Invalid mer_file_type provided: {mer_file_type}.")
        filename = f"EUC_MER_{filename}_TILE{tile_index}.fits"
        outfn = outpath / filename
        if verbose:
            print(
                f"Downloading {mer_file_type} file for tile {tile_index}, instrument {instrument} and filter {filter}"
            )
            print(f"Saving as {outfn}")
            print(params_dict)
        self.data_login()
        if not self.dry_run:
            if not outfn.exists() or self.overwrite:
                self.data_tap.load_data(params_dict=params_dict, output_file=outfn)
            else:
                print(f"File already exists, skipping: {outfn}")

    def download_calibrated_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        instrument_folder = self.get_instrument_folder(instrument)
        file_info = self.get_calibrated_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        outpath = Path(outpath, instrument_folder, f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_sir_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all SIR files for a Euclid observation."""
        file_info = self.get_sir_files_for_observation(obs_id)
        outpath = Path(outpath, "SIR", f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_raw_files_for_observation(
        self,
        obs_id,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter."""
        file_info = self.get_raw_files_for_observation(
            obs_id, instrument=instrument, filter=filter
        )
        outpath = Path(outpath, "RAW", f"{obs_id:n}")
        self.download_files(file_info, outpath=outpath, verbose=verbose)
        return file_info

    def download_files_for_tile(
        self,
        tile_index,
        outpath,  # the base folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        mer_file_type="STK",  # STK, BKG, RMS or FLAG
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all files for a Euclid MER tile, optionally restricted by instrument or filter.

        By default this gets the background subtracted image, but specifying `mer_file_type` can download the
        background model (BKG), RMS or FLAG images.
        """
        file_info = self.get_files_for_tile(
            tile_index, instrument=instrument, filter=filter
        )
        instrument_folder = self.get_instrument_folder(instrument)
        outpath = Path(outpath, "MER", f"{tile_index:n}", instrument_folder)
        filenames = self.download_mosaic_files(
            file_info, outpath=outpath, mer_file_type=mer_file_type, verbose=verbose
        )
        file_info["file_name"] = filenames
        return file_info

    def download_files_for_target(
        self,
        ra,  # RA of the target, as an angular quantity or in decimal degrees
        dec,  # Dec of the target, as an angular quantity or in decimal degrees
        radius=1
        / 60,  # radius of the target, as an angular quantity or in decimal degrees
        fully_contained=True,  # if False, the target region only needs to intersect with the observation footprint
        outpath=None,  # the folder in which to save the downloaded files
        instrument=None,  # None, NISP or VIS
        filter=None,  # None, VIS, NIR_Y, NIR_J or NIR_H
        file_type="CAL",  # CAL, MER
        mer_file_type="STK",  # STK, BKG, RMS or FLAG, only for file_type="MER"
        verbose=True,  # print information to the screen
    ):  #  returns a table of file information
        """Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter."""
        file_info = []
        if file_type == "CAL":
            obs_ids = self.find_observations_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for obs_id in obs_ids:
                if verbose:
                    print(f"Downloading files for observation id {obs_id}")
                obs_file_info = self.download_calibrated_files_for_observation(
                    obs_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                    verbose=verbose,
                )
                file_info.append(obs_file_info)
        elif file_type == "MER":
            tile_ids = self.find_tiles_for_target(
                ra, dec, radius, fully_contained=fully_contained
            )
            for tile_id in tile_ids:
                if verbose:
                    print(f"Downloading files for tile index {tile_id}")
                tile_file_info = self.download_files_for_tile(
                    tile_id,
                    outpath=outpath,
                    instrument=instrument,
                    filter=filter,
                    mer_file_type=mer_file_type,
                    verbose=verbose,
                )
                file_info.append(tile_file_info)
        if len(file_info) > 0:
            return table.vstack(file_info)

### Calibrated files

In [ ]:
show_doc(DataAccess.find_observations_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L155){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_observations_for_target

>      DataAccess.find_observations_for_target (ra, dec,
>                                               radius=0.016666666666666666,
>                                               fully_contained=True)

*Obtain a list of survey obs_ids for observations that entirely contain or intersect the specified target region.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, as an angular quantity or in decimal degrees |
| dec |  |  | Dec of the target, as an angular quantity or in decimal degrees |
| radius | float | 0.016666666666666666 |  |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |

In [ ]:
show_doc(DataAccess.get_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L192){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_calibrated_files_for_observation

>      DataAccess.get_calibrated_files_for_observation (obs_id, instrument=None,
>                                                       filter=None)

*Obtain calibrated file information for obs_id, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  | observation_id for which to find files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |

In [ ]:
show_doc(DataAccess.download_calibrated_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L385){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_calibrated_files_for_observation

>      DataAccess.download_calibrated_files_for_observation (obs_id, outpath,
>                                                            instrument=None,
>                                                            filter=None,
>                                                            verbose=True)

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  |  |
| outpath |  |  | the base folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| verbose | bool | True | print information to the screen |

In [ ]:
show_doc(DataAccess.download_files_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L465){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_files_for_target

>      DataAccess.download_files_for_target (ra, dec,
>                                            radius=0.016666666666666666,
>                                            fully_contained=True, outpath=None,
>                                            instrument=None, filter=None,
>                                            file_type='CAL',
>                                            mer_file_type='STK', verbose=True)

*Download all calibrated files for Euclid observations covering a target, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, as an angular quantity or in decimal degrees |
| dec |  |  | Dec of the target, as an angular quantity or in decimal degrees |
| radius | float | 0.016666666666666666 |  |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |
| outpath | NoneType | None | the folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| file_type | str | CAL | CAL, MER |
| mer_file_type | str | STK | STK, BKG, RMS or FLAG, only for file_type="MER" |
| verbose | bool | True | print information to the screen |

### Mosaic files

In [ ]:
show_doc(DataAccess.find_tiles_for_target)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L174){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.find_tiles_for_target

>      DataAccess.find_tiles_for_target (ra, dec, radius=0.016666666666666666,
>                                        fully_contained=True)

*Obtain a list of survey MER tile_indexes for tiles that entirely contain or intersect the specified target region.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ra |  |  | RA of the target, as an angular quantity or in decimal degrees |
| dec |  |  | Dec of the target, as an angular quantity or in decimal degrees |
| radius | float | 0.016666666666666666 |  |
| fully_contained | bool | True | if False, the target region only needs to intersect with the observation footprint |

In [ ]:
show_doc(DataAccess.get_files_for_tile)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L240){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_files_for_tile

>      DataAccess.get_files_for_tile (tile_index, instrument=None, filter=None)

*Obtain mosaic file information for tile_index, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| tile_index |  |  | tile_index for which to find files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |

In [ ]:
show_doc(DataAccess.download_files_for_tile)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L435){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_files_for_tile

>      DataAccess.download_files_for_tile (tile_index, outpath, instrument=None,
>                                          filter=None, mer_file_type='STK',
>                                          verbose=True)

*Download all files for a Euclid MER tile, optionally restricted by instrument or filter.

By default this gets the background subtracted image, but specifying `mer_file_type` can download the
background model (BKG), RMS or FLAG images.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| tile_index |  |  |  |
| outpath |  |  | the base folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| mer_file_type | str | STK | STK, BKG, RMS or FLAG |
| verbose | bool | True | print information to the screen |

### SIR files

In [ ]:
show_doc(DataAccess.get_sir_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L210){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_sir_files_for_observation

>      DataAccess.get_sir_files_for_observation (obs_id)

*Obtain SIR file information for obs_id.*

|    | **Details** |
| -- | ----------- |
| obs_id | observation_id for which to find files |

In [ ]:
show_doc(DataAccess.download_sir_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L407){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_sir_files_for_observation

>      DataAccess.download_sir_files_for_observation (obs_id, outpath,
>                                                     verbose=True)

*Download all SIR files for a Euclid observation.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  |  |
| outpath |  |  | the base folder in which to save the downloaded files |
| verbose | bool | True | print information to the screen |

### Raw files

In [ ]:
show_doc(DataAccess.get_raw_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L221){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.get_raw_files_for_observation

>      DataAccess.get_raw_files_for_observation (obs_id, instrument=None,
>                                                filter=None)

*Obtain raw file information for obs_id, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  | observation_id for which to find files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, "" for VIS, OPEN for grism, CLOSED for slew dark, Y, J or H |

In [ ]:
show_doc(DataAccess.download_raw_files_for_observation)

---

[source](https://github.com/IntraClusterLight/nicl/blob/main/nicl/euclid/data_access.py#L419){target="_blank" style="float:right; font-size:smaller"}

### DataAccess.download_raw_files_for_observation

>      DataAccess.download_raw_files_for_observation (obs_id, outpath,
>                                                     instrument=None,
>                                                     filter=None, verbose=True)

*Download all calibrated files for a Euclid observation, optionally restricted by instrument or filter.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| obs_id |  |  |  |
| outpath |  |  | the base folder in which to save the downloaded files |
| instrument | NoneType | None | None, NISP or VIS |
| filter | NoneType | None | None, VIS, NIR_Y, NIR_J or NIR_H |
| verbose | bool | True | print information to the screen |

### Example

The following demonstrates use of the DataAccess class. This would normally first be imported using `from nicl.euclid.data_access import DataAccess`.

To actually download files, remove `dry_run=True`.

In [ ]:
release_name = "Q1_R1"

da = DataAccess(release_name=release_name, dry_run=True)

We will download the files into the folder structure at the default location.

In [ ]:
outpath = default_data_path(release_name)

#### Calibrated frames

In [ ]:
da.find_all_observations()

INFO: OK [astroquery.utils.tap.core]


array([2681, 2682, 2683, 2684, 2685, 2686, 2687, 2688, 2689, 2690, 2691,
       2692, 2693, 2694, 2695, 2696, 2697, 2698, 2699, 2700, 2701, 2702,
       2703, 2704, 2705, 2706, 2707, 2708, 2709, 2710, 2711, 2712, 2713,
       2714, 2715, 2716, 2717, 2718, 2719, 2720, 2721, 2722, 2723, 3015,
       3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026,
       3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 3036, 3037,
       3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589,
       3590, 3591, 3592, 3593, 3594, 3595, 3596, 3597, 3598, 3599, 3600,
       3601, 3602, 3603, 3604, 3605, 3606, 3607, 3608, 3609, 3610, 3611,
       3612, 3613, 3614, 3615, 3616, 3617, 3618, 3619, 3620, 3621, 3622,
       3623, 3624])

In [ ]:
da.find_observations_for_target(ra=268.6647, dec=68.0564)

INFO: OK [astroquery.utils.tap.core]


array([2683])

In [ ]:
da.find_observations_for_target(
    ra=268.6647, dec=68.0564, radius=1.0, fully_contained=False
)

INFO: OK [astroquery.utils.tap.core]


array([2682, 2683, 2684, 2685, 2686, 2693, 2694, 2695])

In [ ]:
da.get_calibrated_files_for_observation(2683, instrument="NISP", filter="NIR_H")

INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2683,EUC_NIR_W-CAL-IMAGE_H-2683-1_20240930T184607.455806Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-2_20240930T184607.453358Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-0_20240930T184607.344746Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-3_20240930T184607.529026Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_calibrated_files_for_observation(
    2683, instrument="NISP", filter="NIR_H", outpath=outpath
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2683,EUC_NIR_W-CAL-IMAGE_H-2683-0_20240930T184607.344746Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-3_20240930T184607.529026Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-2_20240930T184607.453358Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-1_20240930T184607.455806Z.fits,NISP,NIR_H,87.2448


In [ ]:
da.download_files_for_target(
    ra=268.6647,
    dec=68.0564,
    instrument="NISP",
    filter="NIR_H",
    file_type="CAL",
    outpath=outpath,
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name,instrument_name,filter_name,duration
str255,str255,str255,str255,float64
2683,EUC_NIR_W-CAL-IMAGE_H-2683-0_20240930T184607.344746Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-3_20240930T184607.529026Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-2_20240930T184607.453358Z.fits,NISP,NIR_H,87.2448
2683,EUC_NIR_W-CAL-IMAGE_H-2683-1_20240930T184607.455806Z.fits,NISP,NIR_H,87.2448


#### MER tiles

::: {.callout-warning}
Note that currently (at 2024-11-20) there does not appear to be any way to programatically download the MER auxilliary files, e.g., BKG, RMS, FLAG. The documented methods are broken. Instead, to download these files, use a browser to search the [SAS](https://easidr.esac.esa.int/sas/) and use the bulk download link on the results page.
:::

In [ ]:
da.find_all_tiles()

INFO: OK [astroquery.utils.tap.core]


array([102018211, 102018212, 102018213, 102018664, 102018665, 102018666,
       102018667, 102018668, 102018669, 102018670, 102018671, 102019122,
       102019123, 102019124, 102019125, 102019126, 102019127, 102019128,
       102019129, 102019130, 102019131, 102019585, 102019586, 102019587,
       102019588, 102019589, 102019590, 102019591, 102019592, 102019593,
       102019594, 102019595, 102019596, 102020053, 102020054, 102020055,
       102020056, 102020057, 102020058, 102020059, 102020060, 102020061,
       102020062, 102020063, 102020064, 102020065, 102020066, 102020527,
       102020528, 102020529, 102020530, 102020531, 102020532, 102020533,
       102020534, 102020535, 102020536, 102020537, 102020538, 102020539,
       102020540, 102020541, 102020542, 102021006, 102021007, 102021008,
       102021009, 102021010, 102021011, 102021012, 102021013, 102021014,
       102021015, 102021016, 102021017, 102021018, 102021019, 102021020,
       102021021, 102021490, 102021491, 102021492, 

In [ ]:
da.find_tiles_for_target(ra=268.6647, dec=68.0564)

INFO: OK [astroquery.utils.tap.core]


array([102160336])

In [ ]:
da.find_tiles_for_target(ra=268.6647, dec=68.0564, radius=1.0, fully_contained=False)

INFO: OK [astroquery.utils.tap.core]


array([102159773, 102159774, 102159775, 102159776, 102160056, 102160057,
       102160058, 102160059, 102160060, 102160334, 102160335, 102160336,
       102160337, 102160338, 102160606, 102160607, 102160608, 102160609,
       102160873])

In [ ]:
da.get_files_for_tile(102160336, instrument="NISP", filter="NIR_H")

INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str255,int64,str255,str255,str255,str255,float64,float64,str255
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,2635,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str82,int64,str255,str255,str255,str255,float64,float64,str255
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,2635,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath, mer_file_type="BKG"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str75,int64,str255,str255,str255,str255,float64,float64,str255
102160336,EUC_MER_BGMOD-NIR-H_TILE102160336-2196A9_20241024T223003.109129Z_00.00.fits,2635,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_tile(
    102160336, instrument="NISP", filter="NIR_H", outpath=outpath, mer_file_type="RMS"
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str80,int64,str255,str255,str255,str255,float64,float64,str255
102160336,EUC_MER_MOSAIC-NIR-H-RMS_TILE102160336-43125E_20241024T215349.991747Z_00.00.fits,2635,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


In [ ]:
da.download_files_for_target(
    ra=268.6647,
    dec=68.0564,
    instrument="NISP",
    filter="NIR_H",
    file_type="MER",
    outpath=outpath,
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


tile_index,file_name,mosaic_product_oid,instrument_name,filter_name,category,second_type,ra,dec,technique
int64,str82,int64,str255,str255,str255,str255,float64,float64,str255
102160336,EUC_MER_BGSUB-MOSAIC-NIR-H_TILE102160336-D13266_20241024T223003.109308Z_00.00.fits,2635,NISP,NIR_H,SCIENCE,SKY,269.1519708,68.0,IMAGE


#### SIR frames

These are the calibrated, but not stacked, spectroscopic frames, which are used in the persistence correction.

In [ ]:
da.get_sir_files_for_observation(2683)

INFO: OK [astroquery.utils.tap.core]


observation_id,file_name
str255,str255
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_2024-10-31T17:49:32.328623Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_2024-10-31T17:46:48.124158Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_2024-10-31T17:54:23.602495Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_2024-10-31T17:48:35.847689Z.fits


In [ ]:
da.download_sir_files_for_observation(2683, outpath=outpath)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


observation_id,file_name
str255,str255
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11837_0_RGS000_0_2024-10-31T17:48:35.847689Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11840_3_RGS180_0_2024-10-31T17:49:32.328623Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11838_1_RGS180_4_2024-10-31T17:46:48.124158Z.fits
2683,EUC_SIR_W-SCIFRM_BKGSUB_2683_11839_2_RGS000_-4_2024-10-31T17:54:23.602495Z.fits


#### Raw frames

For example, we can download the raw slew darks.

In [ ]:
da.get_raw_files_for_observation(2683, instrument="NISP", filter="CLOSED")

INFO: OK [astroquery.utils.tap.core]


file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
str255,int64,int64,str255,str255,str255,str255,str255,str255,str255,float64,float64,object,float64
EUC_LE1_NISP-02683-1-D_20240717T192725.000000Z_01_05_01.00.fits,2357,2683,NISP,SURVEY,CLOSED,CALIBRATION,OPEN,CALBLOCK-F-003,0,268.75735383,68.24852514,2024-07-17T19:44:42.161,87.2448


In [ ]:
da.download_raw_files_for_observation(
    2683, instrument="NISP", filter="CLOSED", outpath=outpath
)

INFO: OK [astroquery.utils.tap.core]
INFO: OK [astroquery.utils.tap.core]


file_name,rawframe_oid,observation_id,instrument_name,data_set_release,filter_name,observation_mode,grism_wheel_pos,cal_block_id,cal_block_variant,ra,dec,obs_time_utc,exposure_time
str255,int64,int64,str255,str255,str255,str255,str255,str255,str255,float64,float64,object,float64
EUC_LE1_NISP-02683-1-D_20240717T192725.000000Z_01_05_01.00.fits,2357,2683,NISP,SURVEY,CLOSED,CALIBRATION,OPEN,CALBLOCK-F-003,0,268.75735383,68.24852514,2024-07-17T19:44:42.161,87.2448


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()